# Team Odysseus — DataSprint 2026## Predicting Financial Status of Kenyan Adults**Objective:** Classify Kenyan adults' financial status into 3 classes: Improved, Stayed the same, Worsened.**Dataset:** 2024 FinAccess Household Survey — 20,848 observations, 28 raw features.**Metric:** Weighted F1-Score.**Class Distribution:** Worsened 52.6% | Stayed same 26.9% | Improved 20.5%

## 1. Setup & Imports

In [ ]:
import pandas as pdimport numpy as npimport warningswarnings.filterwarnings('ignore')from sklearn.model_selection import train_test_split, StratifiedKFoldfrom sklearn.metrics import classification_report, f1_score, ConfusionMatrixDisplayfrom sklearn.utils.class_weight import compute_sample_weightfrom sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifierfrom sklearn.linear_model import LogisticRegressionfrom sklearn.preprocessing import LabelEncoderfrom xgboost import XGBClassifierfrom lightgbm import LGBMClassifierfrom catboost import CatBoostClassifierfrom imblearn.combine import SMOTETomekimport optunaoptuna.logging.set_verbosity(optuna.logging.WARNING)import shapimport matplotlibmatplotlib.use('Agg')import matplotlib.pyplot as pltimport joblib

## 2. Data CleaningClean the raw FinAccess 2024 survey data: remove duplicates, fix education level, handle MNAR missing values in barriers_bank, standardize text.

In [ ]:
# --- Data Cleaning ---df = pd.read_csv('finaccess2024_datasprint.csv')print(f"Loaded raw data: {df.shape[0]} rows, {df.shape[1]} columns")# Drop duplicatesdupes_before = df.duplicated().sum()df = df.drop_duplicates()print(f"Duplicates: Found {dupes_before}, dropped. Rows now: {df.shape[0]}")# Fix education_leveldf['education_level'] = df['education_level'].str.replace('"', '', regex=False).str.strip()df['education_level'] = df['education_level'].replace('None', 'No formal education')junk_education = ['Refused to Answer (DO NOT READ OUT)', "Don't know (DO NOT READ OUT)", 'Other (Specify)', '95']rows_before = len(df)df = df[~df['education_level'].isin(junk_education)]print(f"education_level: Dropped {rows_before - len(df)} junk rows.")# Fix marital_statusjunk_marital = ["Don't know   (DO NOT READ OUT)", "Refused to Answer(DO NOT READ OUT)"]rows_before = len(df)df = df[~df['marital_status'].isin(junk_marital)]print(f"marital_status: Dropped {rows_before - len(df)} junk rows.")# Fill barriers_bank NaN (MNAR — people with bank accounts have no barrier)missing_before = df['barriers_bank'].isnull().sum()df['barriers_bank'] = df['barriers_bank'].fillna('No barrier')print(f"barriers_bank: Filled {missing_before} NaN with 'No barrier'.")# Fix barriers_mobile_moneycount_zero = (df['barriers_mobile_money'] == '0').sum()df['barriers_mobile_money'] = df['barriers_mobile_money'].replace('0', 'No barrier')print(f"barriers_mobile_money: Renamed {count_zero} '0' values to 'No barrier'.")# Strip whitespacetext_cols = df.select_dtypes(include='object').columnsfor col in text_cols:    df[col] = df[col].str.strip()df.to_csv('finaccess2024_cleaned.csv', index=False)print(f"\nSaved cleaned data: {df.shape[0]} rows, {df.shape[1]} columns")print(f"Target distribution:\n{df['financial_status'].value_counts(normalize=True).round(3)}")

## 3. Feature Engineering & Preprocessing28 raw features → 67 features after engineering:- Binary encoding for Yes/No columns- Ordinal encoding for Age, education_level, fl_score- Log transforms, ratio features, composite scores- Interaction terms (shock×defaulted, income×formal, etc.)- Frequency encoding for categoricals- Target encoding columns kept for ML pipeline (applied inside CV)

In [ ]:
def load_and_preprocess():    df = pd.read_csv('finaccess2024_cleaned.csv')    assert df.shape[0] == 20848 and df.isnull().sum().sum() == 0    y = df['financial_status']    X = df.drop(columns=['financial_status'])    # Binary encodings    for col in ['defaulted', 'mobile_money_access', 'mobile_ownership_1',                'experienced_shock', 'nfhi_11', 'nfhi_12', 'nfhi_13',                'accessto_13k_1month', 'not_difficult']:        X[col] = (X[col] == 'Yes').astype(int)    for col in ['Savings_formal', 'Savings_informal', 'Loan_formal',                'Loan_informal', 'formal_service_use']:        X[col] = (X[col] == 'Usage').astype(int)    X['location_type'] = (X['location_type'] == 'Urban').astype(int)    X['Sex'] = (X['Sex'] == 'Male').astype(int)    X['has_disability'] = (X['has_disability'] == 'With Disability').astype(int)    # Ordinal encodings    X['Age'] = X['Age'].map({v: i for i, v in enumerate(        ['16-17', '18-25', '26-35', '36-45', '46-55', 'Above 55'])}).fillna(-1)    X['education_level'] = X['education_level'].map({v: i for i, v in enumerate([        'No formal education', 'Some primary', 'Primary completed', 'Some secondary',        'Secondary completed', 'Some technical training after secondary school',        'Completed technical training after secondary school',        'Some university', 'University completed'])}).fillna(-1)    X['fl_score'] = X['fl_score'].map({v: i for i, v in enumerate(        ['None correct', 'One correct', 'Two correct', 'All correct'])}).fillna(-1)    # Engineered features    X['log_income'] = np.log1p(X['monthly_income'])    X['income_per_person'] = X['monthly_income'] / X['household_size'].clip(lower=1)    X['log_income_per_person'] = np.log1p(X['income_per_person'])    X['nfhi_composite'] = X['nfhi_11'] + X['nfhi_12'] + X['nfhi_13']    X['total_formal_products'] = X['Savings_formal'] + X['Loan_formal'] + X['formal_service_use']    X['total_informal_products'] = X['Savings_informal'] + X['Loan_informal']    X['total_products'] = X['total_formal_products'] + X['total_informal_products']    X['resilience_score'] = X['accessto_13k_1month'] + X['not_difficult'] + (1 - X['defaulted'])    X['shock_vulnerable'] = ((X['experienced_shock'] == 1) & (X['resilience_score'] <= 1)).astype(int)    X['edu_income_ratio'] = X['education_level'] / (X['log_income'] + 1)    X['prodsum1_per_person'] = X['prodsum1'] / X['household_size'].clip(lower=1)    X['age_x_shock'] = X['Age'] * X['experienced_shock']    X['urban_x_formal'] = X['location_type'] * X['total_formal_products']    X['digital_access'] = X['mobile_money_access'] * X['mobile_ownership_1']    X['financial_capability'] = X['education_level'] * X['fl_score']    X['shock_no_resilience'] = X['experienced_shock'] * (1 - X['accessto_13k_1month'])    X['income_quantile'] = pd.qcut(X['monthly_income'], q=5, labels=False, duplicates='drop')    age_midpoint_map = {0: 16.5, 1: 21.5, 2: 30.5, 3: 40.5, 4: 50.5, 5: 60}    X['dependency_ratio'] = X['household_size'] / (1 + X['Age'].map(age_midpoint_map).fillna(30.5))    X['no_savings_no_access'] = ((X['Savings_formal'] == 0) & (X['accessto_13k_1month'] == 0)).astype(int)    # Interactions    X['shock_x_defaulted'] = X['experienced_shock'] * X['defaulted']    X['income_x_formal'] = X['log_income'] * X['total_formal_products']    X['age_x_education'] = X['Age'] * X['education_level']    X['disability_x_shock'] = X['has_disability'] * X['experienced_shock']    X['urban_x_income'] = X['location_type'] * X['log_income']    X['shock_x_no_savings'] = X['experienced_shock'] * X['no_savings_no_access']    X['shock_x_income'] = X['experienced_shock'] * X['log_income']    X['defaulted_x_resilience'] = X['defaulted'] * X['resilience_score']    X['formal_x_resilience'] = X['total_formal_products'] * X['resilience_score']    X['income_x_resilience'] = X['log_income_per_person'] * X['resilience_score']    X['age_x_shock_defaulted'] = X['Age'] * X['shock_x_defaulted']    X['education_x_income'] = X['education_level'] * X['log_income']    X['shock_x_disability_x_income'] = X['shock_vulnerable'] * X['log_income']    X['urban_x_formal_x_income'] = X['urban_x_formal'] * X['log_income']    X['household_burden'] = X['household_size'] / (X['log_income'] + 1)    X['savings_gap'] = 3 - X['total_formal_products']    X['access_gap'] = (1 - X['accessto_13k_1month']) + (1 - X['not_difficult'])    X['vulnerability_composite'] = X['experienced_shock'] + X['defaulted'] + (1 - X['accessto_13k_1month']) + X['no_savings_no_access']    # Frequency encoding    for col in ['marital_status', 'barriers_mobile_money', 'barriers_bank']:        freq = X[col].value_counts(normalize=True)        X[col + '_freq'] = X[col].map(freq).fillna(0)    le = LabelEncoder()    y_encoded = le.fit_transform(y)    cat_cols_to_te = ['county', 'marital_status', 'barriers_mobile_money', 'barriers_bank']    return X, y_encoded, list(le.classes_), le, cat_cols_to_te# Load and display shapeX, y, class_names, le_target, cat_cols_to_te = load_and_preprocess()print(f"Features: {X.shape[1]} (before target encoding)")print(f"Classes: {class_names}")print(f"Cat cols for TE: {cat_cols_to_te}")

## 4. Exploratory Data Analysis

In [ ]:
df_eda = pd.read_csv('finaccess2024_cleaned.csv')fig, axes = plt.subplots(1, 3, figsize=(18, 5))# Target distributiondf_eda['financial_status'].value_counts().plot(kind='bar', ax=axes[0], color=['#e53935', '#ff9800', '#43a047'])axes[0].set_title('Financial Status Distribution', fontsize=14)axes[0].set_ylabel('Count')for label in axes[0].get_xticklabels():    label.set_rotation(0)# Income by classdf_eda.boxplot(column='monthly_income', by='financial_status', ax=axes[1])axes[1].set_title('Monthly Income by Financial Status')axes[1].set_xlabel('')# Shock prevalenceshock_rates = df_eda.groupby('financial_status')['experienced_shock'].apply(lambda x: (x=='Yes').mean())shock_rates.plot(kind='bar', ax=axes[2], color=['#43a047', '#ff9800', '#e53935'])axes[2].set_title('Shock Prevalence by Financial Status', fontsize=14)axes[2].set_ylabel('Proportion Experienced Shock')for label in axes[2].get_xticklabels():    label.set_rotation(0)plt.tight_layout()plt.savefig('eda_overview.png', dpi=150)plt.show()print(f"\nKey Stats:")print(f"  Observations: {len(df_eda)}")print(f"  Counties: {df_eda['county'].nunique()}")print(f"  Mobile money: {(df_eda['mobile_money_access']=='Yes').mean()*100:.1f}%")print(f"  Experienced shock: {(df_eda['experienced_shock']=='Yes').mean()*100:.1f}%")print(f"  Defaulted: {(df_eda['defaulted']=='Yes').mean()*100:.1f}%")print(f"  Median income: KES {df_eda['monthly_income'].median():,.0f}")

## 5. ML Pipeline### 5a. Utility Classes & Functions

In [ ]:
class ThresholdWrapper:    def __init__(self, base_model, weights, n_classes):        self.base_model = base_model        self.weights = weights        self.classes_ = np.arange(n_classes)    def predict_proba(self, X):        return self.base_model.predict_proba(X)    def predict(self, X):        return np.argmax(self.predict_proba(X) * self.weights, axis=1)class SoftVotingWrapper:    def __init__(self, models, weights=None):        self.models = models        self.model_weights = weights        self.classes_ = np.arange(3)    def predict_proba(self, X):        probas = [m.predict_proba(X) for m in self.models]        if self.model_weights is not None:            avg = sum(w * p for w, p in zip(self.model_weights, probas)) / sum(self.model_weights)        else:            avg = np.mean(probas, axis=0)        avg = avg / avg.sum(axis=1, keepdims=True)        return avg    def predict(self, X):        return np.argmax(self.predict_proba(X), axis=1)def target_encode_fold(X_tr, y_tr, X_val, cat_cols, worsened_cls):    overall = (y_tr == worsened_cls).mean()    X_tr, X_val = X_tr.copy(), X_val.copy()    for col in cat_cols:        means = pd.Series(y_tr == worsened_cls).groupby(X_tr[col].values).mean()        X_tr[col + '_te'] = X_tr[col].map(means).fillna(overall)        X_val[col + '_te'] = X_val[col].map(means).fillna(overall)    X_tr = X_tr.drop(columns=cat_cols)    X_val = X_val.drop(columns=cat_cols)    return X_tr, X_valdef target_encode_full(X_data, y_data, cat_cols, worsened_cls):    overall = (y_data == worsened_cls).mean()    X_out = X_data.copy()    te_maps = {}    for col in cat_cols:        means = pd.Series(y_data == worsened_cls).groupby(X_data[col].values).mean()        X_out[col + '_te'] = X_out[col].map(means).fillna(overall)        te_maps[col] = {'means': means, 'overall': overall}    X_out = X_out.drop(columns=cat_cols)    return X_out, te_mapsdef optimize_thresholds(probas, y_true, n_classes, n_iter=150):    best_score = f1_score(y_true, np.argmax(probas, axis=1), average='weighted')    best_weights = np.ones(n_classes)    rng = np.random.RandomState(42)    for i in range(n_iter):        scale = max(0.05, 1.5 - i / n_iter)        candidate = best_weights + rng.normal(0, scale * 0.1, n_classes)        candidate = np.clip(candidate, 0.1, 3.0)        score = f1_score(y_true, np.argmax(probas * candidate, axis=1), average='weighted')        if score > best_score:            best_score, best_weights = score, candidate.copy()    for _ in range(n_iter * 3):        candidate = best_weights + rng.normal(0, 0.01, n_classes)        candidate = np.clip(candidate, 0.1, 3.0)        score = f1_score(y_true, np.argmax(probas * candidate, axis=1), average='weighted')        if score > best_score:            best_score, best_weights = score, candidate.copy()    return best_weights, best_score

### 5b. Train/Test Split & Target Encoding

In [ ]:
worsened_cls = list(class_names).index('Worsened')X_train_full, X_test, y_train_full, y_test = train_test_split(    X, y, test_size=0.2, random_state=42, stratify=y)X_train_final, train_te_maps = target_encode_full(X_train_full, y_train_full, cat_cols_to_te, worsened_cls)X_test_final = X_test.copy()for col in cat_cols_to_te:    X_test_final[col + '_te'] = X_test_final[col].map(train_te_maps[col]['means']).fillna(train_te_maps[col]['overall'])X_test_final = X_test_final.drop(columns=cat_cols_to_te)X_all_final, _ = target_encode_full(X, y, cat_cols_to_te, worsened_cls)print(f"{X_train_final.shape[1]} features after target encoding")

### 5c. Optuna Hyperparameter Tuning (CatBoost)

In [ ]:
def cb_objective(trial):    p = {        'iterations': 300,        'learning_rate': trial.suggest_float('lr', 0.02, 0.1, log=True),        'depth': trial.suggest_int('depth', 4, 7),        'l2_leaf_reg': trial.suggest_float('l2', 1.0, 10.0),        'random_strength': trial.suggest_float('rs', 0.5, 3.0),        'bagging_temperature': trial.suggest_float('bt', 0.0, 1.0),        'border_count': 254,        'auto_class_weights': 'Balanced', 'verbose': 0, 'random_seed': 42    }    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)    scores = []    for ti, vi in skf.split(X_train_final, y_train_full):        smt = SMOTETomek(random_state=42)        Xr, yr = smt.fit_resample(X_train_final.iloc[ti], y_train_full[ti])        m = CatBoostClassifier(**p).fit(Xr, yr)        scores.append(f1_score(y_train_full[vi], m.predict(X_train_final.iloc[vi]).ravel(), average='weighted'))    return np.mean(scores)study = optuna.create_study(direction='maximize')study.optimize(cb_objective, n_trials=8)best_cb = study.best_params.copy()best_cb.update({'iterations': 1000, 'auto_class_weights': 'Balanced', 'verbose': 0, 'random_seed': 42,                'learning_rate': best_cb.pop('lr'), 'l2_leaf_reg': best_cb.pop('l2'),                'random_strength': best_cb.pop('rs'), 'bagging_temperature': best_cb.pop('bt')})print(f"Best CatBoost: CV={study.best_value:.4f}")

### 5d. Cross-Validation (5-Fold, SMOTE-Tomek train only)

In [ ]:
def get_base_models():    return [        ("XGB", XGBClassifier(            n_estimators=800, learning_rate=0.039, max_depth=6,            min_child_weight=8, subsample=0.7, colsample_bytree=0.96,            gamma=2.06, reg_alpha=1.71, random_state=42, n_jobs=-1        )),        ("LGBM", LGBMClassifier(            n_estimators=800, learning_rate=0.05, max_depth=6,            num_leaves=40, min_child_samples=20, subsample=0.8,            colsample_bytree=0.7, reg_alpha=0.5, reg_lambda=1.0,            class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1        )),        ("CatBoost", CatBoostClassifier(**best_cb)),        ("GB", GradientBoostingClassifier(            n_estimators=500, learning_rate=0.05, max_depth=5,            subsample=0.8, min_samples_leaf=5, random_state=42        )),    ]skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)base_models = get_base_models()cv_scores = {n: [] for n, _ in base_models}oof_probas = {n: np.zeros((len(y_train_full), len(class_names))) for n, _ in base_models}for fold, (ti, vi) in enumerate(skf.split(X_train_full, y_train_full), 1):    print(f"--- Fold {fold} ---")    Xtr = X_train_full.iloc[ti].copy()    ytr = y_train_full[ti]    Xva = X_train_full.iloc[vi].copy()    yva = y_train_full[vi]    Xtr, Xva = target_encode_fold(Xtr, ytr, Xva, cat_cols_to_te, worsened_cls)    smt = SMOTETomek(random_state=42)    Xtr_r, ytr_r = smt.fit_resample(Xtr, ytr)    fsw = compute_sample_weight('balanced', ytr_r)    for name, model in base_models:        m = type(model)(**model.get_params())        if name == "XGB":            m.fit(Xtr_r, ytr_r, sample_weight=fsw)        else:            m.fit(Xtr_r, ytr_r)        probas = m.predict_proba(Xva)        cv_scores[name].append(f1_score(yva, np.argmax(probas, axis=1), average='weighted'))        oof_probas[name][vi] = probas        print(f"  {name:15s}: F1={cv_scores[name][-1]:.4f}")mean_cv = {k: np.mean(v) for k, v in cv_scores.items()}print("\n--- Mean CV ---")for n in sorted(mean_cv, key=mean_cv.get, reverse=True):    print(f"  {n:15s}: {mean_cv[n]:.4f}")

### 5e. Stacking Meta-Learner & Full Retraining

In [ ]:
# Stacking meta-learner on OOF predictionsmeta_train = np.hstack([oof_probas[n] for n, _ in base_models])meta_lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42, C=0.5)meta_lr.fit(meta_train, y_train_full)# Retrain on full train set with SMOTE-Tomeksmt = SMOTETomek(random_state=42)Xr, yr = smt.fit_resample(X_train_final, y_train_full)sw = compute_sample_weight('balanced', yr)trained = {}for name, model in get_base_models():    print(f"Retraining {name}...")    m = type(model)(**model.get_params())    if name == "XGB":        m.fit(Xr, yr, sample_weight=sw)    else:        m.fit(Xr, yr)    trained[name] = m

### 5f. Test Scores & Blended Ensemble

In [ ]:
# Test scorestest_scores, test_probas = {}, {}for name, model in trained.items():    p = model.predict_proba(X_test_final)    test_probas[name] = p    test_scores[name] = f1_score(y_test, np.argmax(p, axis=1), average='weighted')    print(f"  {name:15s}: F1={test_scores[name]:.4f}")# Stackingmt = np.hstack([trained[n].predict_proba(X_test_final) for n, _ in base_models])test_probas["Stacking"] = meta_lr.predict_proba(mt)test_scores["Stacking"] = f1_score(y_test, meta_lr.predict(mt), average='weighted')print(f"  Stacking        : F1={test_scores['Stacking']:.4f}")# Blended Ensemble - joint blend + threshold search on OOFmodel_names_list = [n for n, _ in base_models]best_blend_score, best_blend_w, best_thresh_w = 0, np.ones(len(model_names_list)) / len(model_names_list), np.ones(len(class_names))threshold_weights = {}rng = np.random.RandomState(42)for i in range(5000):    bw = rng.dirichlet(np.ones(len(model_names_list)))    blended = sum(w * oof_probas[n] for w, n in zip(bw, model_names_list))    blended /= blended.sum(axis=1, keepdims=True)    tw = np.ones(len(class_names)) + rng.normal(0, 0.3, len(class_names))    tw = np.clip(tw, 0.3, 2.5)    preds = np.argmax(blended * tw, axis=1)    s = f1_score(y_train_full, preds, average='weighted')    if s > best_blend_score:        best_blend_score, best_blend_w, best_thresh_w = s, bw.copy(), tw.copy()for _ in range(10000):    bw = best_blend_w + rng.normal(0, 0.02, len(model_names_list))    bw = np.clip(bw, 0.01, None)    bw /= bw.sum()    blended = sum(w * oof_probas[n] for w, n in zip(bw, model_names_list))    blended /= blended.sum(axis=1, keepdims=True)    tw = best_thresh_w + rng.normal(0, 0.02, len(class_names))    tw = np.clip(tw, 0.3, 2.5)    preds = np.argmax(blended * tw, axis=1)    s = f1_score(y_train_full, preds, average='weighted')    if s > best_blend_score:        best_blend_score, best_blend_w, best_thresh_w = s, bw.copy(), tw.copy()print(f"Best blend: {dict(zip(model_names_list, best_blend_w.round(3)))}")print(f"Best thresh: {best_thresh_w.round(3)}, OOF F1: {best_blend_score:.4f}")sv_blend = SoftVotingWrapper([trained[n] for n in model_names_list], weights=list(best_blend_w))blend_proba = sv_blend.predict_proba(X_test_final)blend_f1 = f1_score(y_test, np.argmax(blend_proba * best_thresh_w, axis=1), average='weighted')test_scores["BlendTuned"] = blend_f1trained["BlendTuned"] = ThresholdWrapper(sv_blend, best_thresh_w, len(class_names))test_probas["BlendTuned"] = blend_probathreshold_weights["BlendTuned"] = best_thresh_wprint(f"BlendTuned Test F1: {blend_f1:.4f}")

### 5g. Threshold Optimization (Individual Models)

In [ ]:
for mn in ["CatBoost", "XGB", "LGBM", "GB"]:    bw, bvs = optimize_thresholds(oof_probas[mn], y_train_full, len(class_names), n_iter=150)    tp = test_probas[mn]    tf1 = f1_score(y_test, np.argmax(tp * bw, axis=1), average='weighted')    test_scores[f"{mn}+Threshold"] = tf1    threshold_weights[f"{mn}+Threshold"] = bw    trained[f"{mn}+Threshold"] = ThresholdWrapper(trained[mn], bw, len(class_names))    print(f"  {mn}: weights={bw.round(3)}  Test F1={tf1:.4f}")

### 5h. Final Leaderboard & Evaluation

In [ ]:
final_lb = sorted(test_scores.items(), key=lambda x: x[1], reverse=True)for r, (n, s) in enumerate(final_lb, 1):    print(f"  {r:2d}. {n:35s} F1={s:.4f}{' <<<< WINNER' if r == 1 else ''}")best_name = final_lb[0][0]best_model = trained[best_name]y_pred = best_model.predict(X_test_final)if hasattr(y_pred, 'ravel'):    y_pred = y_pred.ravel()f1_final = f1_score(y_test, y_pred, average='weighted')print(f"\nFINAL: {best_name} — F1={f1_final:.4f}")print(classification_report(y_test, y_pred, target_names=class_names))fig, ax = plt.subplots(figsize=(8, 6))ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=class_names, cmap='Blues', ax=ax)ax.set_title(f'{best_name} (F1={f1_final:.4f})')plt.tight_layout()plt.savefig('confusion_matrix.png', dpi=150)plt.show()

### 5i. Save Pipeline State

In [ ]:
joblib.dump(best_model, 'odysseus_final_model.pkl')te_maps_save = {col: {'means': train_te_maps[col]['means'], 'overall': train_te_maps[col]['overall']} for col in cat_cols_to_te}joblib.dump({    'X_test_final': X_test_final, 'y_test': y_test, 'y_pred': y_pred,    'class_names': class_names,    'trained': {k: v for k, v in trained.items() if not isinstance(v, (ThresholdWrapper, SoftVotingWrapper))},    'test_scores': test_scores, 'mean_cv': mean_cv,    'X_all_final': X_all_final, 'y_full': y,    'te_maps': te_maps_save, 'cat_cols_to_te': cat_cols_to_te,    'worsened_cls': worsened_cls, 'threshold_weights': threshold_weights,    'best_name': best_name, 'meta_lr': meta_lr, 'best_cb_params': best_cb,}, 'pipeline_state.pkl')pd.DataFrame({'True': y_test.ravel(), 'Predicted': y_pred.ravel()}).to_csv('final_predictions.csv', index=False)pd.DataFrame(final_lb, columns=['Model', 'Score']).to_csv('model_leaderboard.csv', index=False)print("Saved artifacts successfully.")

## 6. SHAP Explainability & Financial Vulnerability Index

In [ ]:
state = joblib.load('pipeline_state.pkl')X_test_s = state['X_test_final']X_full_s = state['X_all_final']y_full_s = state['y_full']trained_s = state['trained']threshold_weights_s = state['threshold_weights']best_name_s = state['best_name']df_orig = pd.read_csv('finaccess2024_cleaned.csv')shap_model = trained_s["CatBoost"]print(f"Using CatBoost for SHAP")X_sample = X_test_s.sample(n=min(500, len(X_test_s)), random_state=42)explainer = shap.TreeExplainer(shap_model)shap_values = explainer.shap_values(X_sample)if isinstance(shap_values, np.ndarray) and len(shap_values.shape) == 3:    shap_list = [shap_values[:, :, i] for i in range(len(class_names))]else:    shap_list = shap_valuesfor i, cls in enumerate(class_names):    print(f"  SHAP for '{cls}'...")    fig, ax = plt.subplots(figsize=(12, 8))    shap.summary_plot(shap_list[i], X_sample, max_display=15, show=False)    plt.title(f'What drives "{cls}" predictions', fontsize=14)    plt.tight_layout()    plt.savefig(f'shap_{cls.lower().replace(" ", "_")}.png', dpi=150, bbox_inches='tight')    plt.close()fig, ax = plt.subplots(figsize=(12, 8))shap.summary_plot(shap_list, X_sample, class_names=class_names, max_display=15, show=False, plot_type='bar')plt.title('Feature Importance Across All Classes (SHAP)', fontsize=14)plt.tight_layout()plt.savefig('shap_all_classes.png', dpi=150)plt.close()# Vulnerability Indexproba = shap_model.predict_proba(X_full_s)worsened_idx = class_names.index('Worsened')if best_name_s in threshold_weights_s:    raw_worsened_prob = (proba * threshold_weights_s[best_name_s])[:, worsened_idx]else:    raw_worsened_prob = proba[:, worsened_idx]vulnerability = (raw_worsened_prob / raw_worsened_prob.max() * 100).round(1)df_orig['vulnerability_score'] = vulnerabilityprint(f"Vulnerability: range={vulnerability.min():.1f}-{vulnerability.max():.1f}, mean={vulnerability.mean():.1f}")print(f"High Risk (>70): {(vulnerability>70).sum()} ({(vulnerability>70).mean()*100:.1f}%)")fig, ax = plt.subplots(figsize=(10, 6))ax.hist(vulnerability, bins=50, color='steelblue', edgecolor='white', alpha=0.85)ax.axvline(vulnerability.mean(), color='red', linestyle='--', label=f'Mean: {vulnerability.mean():.1f}')ax.axvline(70, color='orange', linestyle='--', label='High Risk Threshold (70)')ax.set_xlabel('Vulnerability Score (0-100)', fontsize=12)ax.set_ylabel('Number of People', fontsize=12)ax.set_title('Financial Vulnerability Distribution Across Kenya', fontsize=14)ax.legend()plt.tight_layout()plt.savefig('vulnerability_distribution.png', dpi=150)plt.close()# County analysiscounty_stats = df_orig.groupby('county').agg(    population=('vulnerability_score', 'count'),    avg_vulnerability=('vulnerability_score', 'mean'),    worsened_rate=('financial_status', lambda x: (x == 'Worsened').mean()),).round(2)county_stats = county_stats.sort_values('avg_vulnerability', ascending=False)print("\nTop 5 Most Vulnerable Counties:")for c, r in county_stats.head(5).iterrows():    print(f"  {c}: avg={r['avg_vulnerability']:.1f}, worsened={r['worsened_rate']*100:.0f}%")fig, ax = plt.subplots(figsize=(14, 10))colors = ['#d32f2f' if v > 60 else '#ff9800' if v > 50 else '#4caf50' for v in county_stats['avg_vulnerability']]county_stats['avg_vulnerability'].plot(kind='barh', ax=ax, color=colors)ax.set_xlabel('Average Vulnerability Score', fontsize=12)ax.set_title('Financial Vulnerability by County', fontsize=14)ax.axvline(county_stats['avg_vulnerability'].mean(), color='black', linestyle='--', alpha=0.5)ax.invert_yaxis()plt.tight_layout()plt.savefig('county_vulnerability.png', dpi=150)plt.close()

## 7. Policy Intervention Simulation

In [ ]:
state = joblib.load('pipeline_state.pkl')X_all_i = state['X_all_final']trained_i = state['trained']model_int = trained_i["CatBoost"]worsened_idx = class_names.index('Worsened')df_orig = pd.read_csv('finaccess2024_cleaned.csv')def get_vulnerability(X_data):    return model_int.predict_proba(X_data)[:, worsened_idx] * 100baseline = get_vulnerability(X_all_i)interventions = []# 1. Mobile MoneyX_sim = X_all_i.copy()mask = df_orig['mobile_money_access'] == 'No'X_sim.loc[mask, 'mobile_money_access'] = 1X_sim.loc[mask, 'digital_access'] = X_sim.loc[mask, 'mobile_ownership_1']sim = get_vulnerability(X_sim)interventions.append(("Mobile Money\nfor All", baseline[mask].mean(), sim[mask].mean(), mask.sum(), (baseline[mask]>70).sum(), (sim[mask]>70).sum()))# 2. Financial LiteracyX_sim = X_all_i.copy()mask = df_orig['fl_score'].isin(['None correct', 'One correct'])X_sim.loc[mask, 'fl_score'] = 3X_sim.loc[mask, 'financial_capability'] = X_sim.loc[mask, 'education_level'] * 3sim = get_vulnerability(X_sim)interventions.append(("Financial Literacy\nProgram", baseline[mask].mean(), sim[mask].mean(), mask.sum(), (baseline[mask]>70).sum(), (sim[mask]>70).sum()))# 3. Formal SavingsX_sim = X_all_i.copy()mask = df_orig['Savings_formal'] == 'Non-usage'X_sim.loc[mask, 'Savings_formal'] = 1X_sim.loc[mask, 'total_formal_products'] = X_sim.loc[mask, 'total_formal_products'] + 1X_sim.loc[mask, 'total_products'] = X_sim.loc[mask, 'total_products'] + 1X_sim.loc[mask, 'no_savings_no_access'] = 0sim = get_vulnerability(X_sim)interventions.append(("Formal Savings\nAccess", baseline[mask].mean(), sim[mask].mean(), mask.sum(), (baseline[mask]>70).sum(), (sim[mask]>70).sum()))# 4. Shock ProtectionX_sim = X_all_i.copy()mask = df_orig['experienced_shock'] == 'Yes'X_sim.loc[mask, 'experienced_shock'] = 0X_sim.loc[mask, 'shock_vulnerable'] = 0X_sim.loc[mask, 'age_x_shock'] = 0X_sim.loc[mask, 'shock_x_defaulted'] = 0X_sim.loc[mask, 'shock_no_resilience'] = 0X_sim.loc[mask, 'shock_x_no_savings'] = 0X_sim.loc[mask, 'disability_x_shock'] = 0sim = get_vulnerability(X_sim)interventions.append(("Shock Protection\n/ Insurance", baseline[mask].mean(), sim[mask].mean(), mask.sum(), (baseline[mask]>70).sum(), (sim[mask]>70).sum()))# 5. Emergency FundX_sim = X_all_i.copy()mask = df_orig['accessto_13k_1month'] == 'No'X_sim.loc[mask, 'accessto_13k_1month'] = 1X_sim.loc[mask, 'resilience_score'] = X_sim.loc[mask, 'resilience_score'] + 1X_sim.loc[mask, 'shock_no_resilience'] = 0sim = get_vulnerability(X_sim)interventions.append(("Emergency Fund\nAccess", baseline[mask].mean(), sim[mask].mean(), mask.sum(), (baseline[mask]>70).sum(), (sim[mask]>70).sum()))# Plotfig, ax = plt.subplots(figsize=(14, 7))names = [i[0] for i in interventions]befores = [i[1] for i in interventions]afters = [i[2] for i in interventions]x = np.arange(len(names))w = 0.35bars1 = ax.bar(x - w/2, befores, w, label='Before', color='#e53935', alpha=0.85)bars2 = ax.bar(x + w/2, afters, w, label='After', color='#43a047', alpha=0.85)ax.set_ylabel('Average Vulnerability Score', fontsize=12)ax.set_title('Impact of Policy Interventions on Financial Vulnerability', fontsize=14, fontweight='bold')ax.set_xticks(x)ax.set_xticklabels(names, fontsize=10)ax.legend()ax.set_ylim(0, max(befores) + 10)plt.tight_layout()plt.savefig('intervention_impact.png', dpi=150)plt.show()for name, before, after, count, hr_before, hr_after in interventions:    print(f"  {name.replace(chr(10),' ')}: {before:.1f} -> {after:.1f} (delta={after-before:+.1f}), High-risk: {hr_before} -> {hr_after} ({hr_after-hr_before:+d})")

## 8. Persona Profiles & County Rankings

In [ ]:
state = joblib.load('pipeline_state.pkl')trained_v = state['trained']model_v = trained_v["CatBoost"]X_all_v = state['X_all_final']df = pd.read_csv('finaccess2024_cleaned.csv')proba = model_v.predict_proba(X_all_v)vuln = proba[:, class_names.index('Worsened')] * 100df['vuln'] = vulnprint('=== VULNERABILITY INDEX ===')print(f'Range: {vuln.min():.1f} - {vuln.max():.1f}')print(f'Mean: {vuln.mean():.1f}, High risk (>70): {(vuln>70).sum()} ({(vuln>70).mean()*100:.1f}%)')print('\n=== COUNTY RANKINGS (TOP 10 MOST VULNERABLE) ===')cs = df.groupby('county').agg(n=('vuln','count'), avg=('vuln','mean'), worsened=('financial_status', lambda x: (x=='Worsened').mean())).sort_values('avg', ascending=False)for c, r in cs.head(10).iterrows():    print(f"  {c:20s} avg={r['avg']:.1f}  worsened={r['worsened']*100:.1f}%  n={int(r['n'])}")for label in ['Worsened', 'Improved', 'Stayed the same']:    subset = df[df['financial_status'] == label]    print(f"\n--- {label.upper()} PERSONA ---")    print(f"  Age: {subset['Age'].mode().iloc[0]}, Sex: {subset['Sex'].mode().iloc[0]}")    print(f"  Median income: KES {subset['monthly_income'].median():,.0f}")    print(f"  Shock: {(subset['experienced_shock']=='Yes').mean()*100:.1f}%, Defaulted: {(subset['defaulted']=='Yes').mean()*100:.1f}%")    print(f"  Formal savings: {(subset['Savings_formal']=='Usage').mean()*100:.1f}%, Can access 13k: {(subset['accessto_13k_1month']=='Yes').mean()*100:.1f}%")

## 9. Key Findings & Recommendations1. **Best Model:** CatBoost+Threshold (Weighted F1 ~0.5485)2. **#1 Driver:** Geographic location + economic shocks without safety nets3. **#1 Intervention:** Shock protection (insurance/safety nets) — rescues 1,104 from high-risk4. **#2 Intervention:** Emergency fund access (KES 13k within 30 days)5. **#3 Intervention:** Formal savings access6. **Paradox:** Mobile money expansion & financial literacy alone may increase vulnerability without structural resilience7. **Geographic targeting:** Tana River, Homabay, Kisumu need urgent intervention (FVI > 68)